# Event Study: France-Only Sample (US Election)

Earlier, France-only version of the event study, used to develop the methodology before extending it to the combined France/Italy/Spain sample.

In [ ]:
import pandas as pd
import pandas_datareader.data as reader

import statsmodels.api as sm
from scipy.stats import ttest_1samp
from scipy.stats import t, norm
import math
import matplotlib.pyplot as plt

import eikon as ek
import os
ek.set_app_key(os.environ["EIKON_APP_KEY"])  # set your own Refinitiv Eikon app key

import yfinance as yf
from datetime import timedelta

In [ ]:
#Data acquisition
stock_returns = pd.read_excel('FR_test_T_election.xlsx', index_col=0, parse_dates=True)
stock_returns = stock_returns / 100
start_date = stock_returns.index.min().strftime('%Y-%m-%d')
end_date = stock_returns.index.max().strftime('%Y-%m-%d')

cac = ek.get_timeseries('.FCHI', fields='CLOSE', start_date=start_date, end_date=end_date, interval='daily')
cac['Market'] = cac['CLOSE'].pct_change()
cac = cac[['Market']].dropna()
cac = cac.sort_index()

stock_attributes = pd.read_excel('FR_stock_attributes.xlsx')
stock_attributes = stock_attributes.set_index('Instrument')
stock_attributes.columns = ['Industry', 'ESG Score']  # rename for clarity

def classify_green_brown(grade):
    if grade in ['A+', 'A', 'A-']:
        return 'Green'
    elif grade in ['D+', 'D', 'D-']:
        return 'Brown'
    else:
        return 'Unknown/Intermediate'  # optionally use this category

# Apply classification
stock_attributes['Environmental Category'] = stock_attributes['ESG Score'].apply(classify_green_brown)


factors = pd.read_csv('famafrench.csv', index_col=0, parse_dates=True)

In [ ]:
#Data clean-up
stock_returns = stock_returns.loc[:, stock_returns.isna().mean() < 0.5]
stock_returns = stock_returns.loc[:, (stock_returns == 0).mean() < 0.5]
stock_returns = stock_returns.loc[:, stock_returns.nunique(dropna=True) > 1]
stock_returns = stock_returns.fillna(0)
stock_returns = stock_returns.sort_index()

cac = cac[['Market']].dropna()


In [ ]:
#OLS Regression using CAPM

combined_returns = stock_returns.join(cac, how='inner')
combined_returns.index = pd.to_datetime(combined_returns.index)


event_date = pd.to_datetime("2024-11-05")
window = 5  # days before and after

# Get list of dates to exclude
event_dates = pd.date_range(event_date - timedelta(days=window), event_date + timedelta(days=window))


estimation_data = combined_returns[(~combined_returns.index.isin(event_dates)) & ~(combined_returns.index >= event_date + timedelta(days=window))]
event_data = combined_returns[combined_returns.index.isin(event_dates)]


capm_df = []

for stock in estimation_data.columns.drop('Market'):
    y = estimation_data[stock]
    X = sm.add_constant(estimation_data['Market'])
    model = sm.OLS(y, X.astype(float)).fit()
    
    capm_df.append({
        'Stock': stock,
        'Alpha': model.params['const'],
        'Beta': model.params['Market'],
        'R-squared': model.rsquared,
        'p-value (Beta)': model.pvalues['Market']
    })

capm_df = pd.DataFrame(capm_df)
#capm_df[capm_df['Stock'] == 'LVMH.PA']
capm_df = capm_df.set_index('Stock')

In [ ]:
#Expected return calculation for CAPM

abnormal_returns_event = pd.DataFrame(index=event_data.index)

for index, row in capm_df.iterrows():
    alpha = row['Alpha']
    beta = row['Beta']
    
    abnormal_returns_event[index] = event_data[index] - (alpha + beta * event_data['Market'])

abnormal_returns_estimation = pd.DataFrame(index=estimation_data.index)

for index, row in capm_df.iterrows():
    alpha = row['Alpha']
    beta = row['Beta']
    
    abnormal_returns_estimation[index] = estimation_data[index] - (alpha + beta * estimation_data['Market'])

ticker_to_sector = stock_attributes['Environmental Category'].to_dict()

# Create MultiIndex for columns: (Sector, Ticker)
new_columns = [
    (ticker_to_sector.get(ticker), ticker)
    for ticker in abnormal_returns_estimation.columns
]

# Set the new MultiIndex
abnormal_returns_estimation.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])
abnormal_returns_event.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])


In [ ]:
CAR = abnormal_returns_event.sum()
CAAR = CAR.mean()
AAR = abnormal_returns_event.mean(axis=1)

#print("CAR per stock:")
CAR.dropna(inplace=True)
#print(CAR.sort_values(ascending=False))
print(f"\nCAAR ([-5, +5] window): {CAAR:.4%}")

print((AAR.mean()/AAR.std())*math.sqrt(AAR.count()))

In [ ]:
t_stat, p_val = ttest_1samp(AAR, 0)
print(f"T-stat: {t_stat:.2f}, p-value: {p_val:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
print(AAR.index)
# Plot each stock line

# Event marker
smoothed = AAR[AAR.index.isin(event_dates)].rolling(window=2, center=True).mean()  # 3-day window

plt.plot(AAR[AAR.index.isin(event_dates)], label="Raw AAR", alpha=0.4)
plt.plot(smoothed, label="Smoothed AAR (3-day MA)", color='blue')
plt.axhline(0, color='black', linestyle='--')
plt.legend()
plt.title("Average Abnormal Return (AAR) with Smoothing")
plt.xlabel("Event Day")
plt.ylabel("AAR")
plt.grid(True)


In [ ]:
#Patell test

def run_patell_test(abnormal_returns_estimation, abnormal_returns_event, estimation_data, event_data, event_date, market_column):
    """
    Computes the Patell and BMP test statistics and p-values.

    Parameters:
    - abnormal_returns_estimation: pd.DataFrame of ARs in estimation window (rows: dates, cols: firms)
    - abnormal_returns_event: pd.DataFrame of ARs in event window (same structure)
    - estimation_data: pd.DataFrame containing at least 'Market' column during estimation
    - event_data: pd.DataFrame containing at least 'Market' column for the event date
    - event_date: str (e.g. '2025-01-20') — the event date

    Returns:
    - dict with Patell z, Patell p-value, BMP z, BMP p-value
    """

    market_mean = estimation_data[market_column].mean()
    M_i = abnormal_returns_estimation.count()

    '''
    # std_AR_i_0: event-day standard error of abnormal return per firm
    std_AR_i_0 = abnormal_returns_estimation.std() * (
        1
        + 1 / abnormal_returns_estimation.count()
        + (event_data.loc[event_date, market_column] - market_mean) ** 2 /
          ((estimation_data[market_column] - market_mean) ** 2).sum()
    ) ** 0.5
    
    # Standardized AR (SAR) on event day
    SAR_0 = abnormal_returns_event.loc[event_date] / std_AR_i_0

    # ASAR: sum of standardized ARs
    ASAR_0 = SAR_0.sum()

    #denominator
    S2_ASAR = ((M_i - 2) / (M_i - 4)).sum()

     # Patell z ~ t
    z_patell = ASAR_0 / S2_ASAR**0.5
    p_patell = 2 * (1 - norm.cdf(abs(z_patell)))
    '''
    std_AR_i = pd.DataFrame(index=event_data.index, columns=abnormal_returns_estimation.columns)

    # Loop over event days to fill in the standard error
    for date in event_data.index:
        std_AR_i.loc[date] = abnormal_returns_estimation.std() * (
        1
        + 1 / M_i
        + ((event_data[market_column] - market_mean) ** 2).loc[date] / ((estimation_data[market_column] - market_mean) ** 2).sum()
        ) ** 0.5
    
    SAR = abnormal_returns_event / std_AR_i
    CSAR = SAR.sum()

    std_CSAR = (len(abnormal_returns_event)*((M_i - 2)/(M_i - 4)))**0.5

    z_patell_CAAR = CSAR.count()**(-0.5)*((CSAR/std_CSAR).sum())
    p_patell_CAAR = 2 * (1 - norm.cdf(abs(z_patell_CAAR)))

    corr_matrix = abnormal_returns_estimation.corr()

    # Adjusting for Kolari and Pynnönen (2010) correction
    N = len(corr_matrix)
    r_bar = (corr_matrix.values.sum() - N) / (N * (N - 1))
    z_adj_patell_CAAR = z_patell_CAAR*(((1 - r_bar) / (1 + (N - 1) * r_bar))**0.5)

    
    results = {
        'Test': ['Patell (CAAR)', 'Adj Patell (CAAR)'],
        'z-statistic': [z_patell_CAAR, z_adj_patell_CAAR],
        'p-value': [p_patell_CAAR, 2 * (1 - norm.cdf(abs(z_adj_patell_CAAR)))]
    }

    return pd.DataFrame(results).set_index('Test')



In [ ]:
#CSect T
AAR_0 = abnormal_returns_event.loc[event_date].mean()

print(abnormal_returns_event.loc[event_date].count())
t1 = (abnormal_returns_event.loc[event_date].count() ** 0.5) * AAR_0 / abnormal_returns_event.loc[event_date].std()

p_values = 2 * (1 - t.cdf(abs(t1), df=500))

print(t1)
print(p_values)


In [ ]:
def run_bmp_test(abnormal_returns_estimation, abnormal_returns_event, estimation_data, event_data, market_column):
    """
    Computes the BMP and adjusted BMP test statistics and p-values for cumulative abnormal returns (CAR).

    Parameters:
    - abnormal_returns_estimation: pd.DataFrame of ARs in estimation window (rows: dates, cols: firms)
    - abnormal_returns_event: pd.DataFrame of ARs in event window (same structure)
    - estimation_data: pd.DataFrame containing market returns during estimation window (with market_col)
    - event_data: pd.DataFrame containing market returns during event window (with market_col)
    - market_col: str, name of the market return column (default 'Market')

    Returns:
    - dict with BMP t-statistic, BMP p-value, adjusted BMP t-statistic, adjusted BMP p-value
    """

    market_mean = estimation_data[market_column].mean()

    # Calculate CAR (sum of abnormal returns in event window) per firm
    CAR = abnormal_returns_event.sum()

    # Calculate standard deviation for CAR using estimation data and market variance
    std_CAR = abnormal_returns_estimation.std() * (
        abnormal_returns_event.count()
        + abnormal_returns_event.count() / abnormal_returns_estimation.count()
        + ((event_data[market_column] - market_mean) ** 2).sum() / ((estimation_data[market_column] - market_mean) ** 2).sum()
    ) ** 0.5

    # Standardized cumulative abnormal returns (SCAR)
    SCAR = CAR / std_CAR

    # BMP t-statistic (mean SCAR scaled by its sample std and sample size)
    t_BMP = (SCAR.mean() / SCAR.std()) * (SCAR.count() ** 0.5)

    # Correlation matrix of estimation window ARs across firms
    corr_matrix = abnormal_returns_estimation.corr()

    # Number of firms
    N = len(corr_matrix)

    # Average correlation (excluding diagonal)
    r_bar = (corr_matrix.values.sum() - N) / (N * (N - 1))

    # Adjust BMP t-statistic for cross-sectional correlation
    t_BMP_adj = t_BMP * ((1 - r_bar) / (1 + (N - 1) * r_bar)) ** 0.5

    # p-values for BMP and adjusted BMP (two-tailed t-test)
    df = SCAR.count() - 1
    p_BMP = 2 * (1 - t.cdf(abs(t_BMP), df=df))
    p_BMP_adj = 2 * (1 - t.cdf(abs(t_BMP_adj), df=df))

    results = {
        'Test': ['BMP (CAAR)', 'Adj BMP (CAAR)'],
        't-statistic': [t_BMP, t_BMP_adj],
        'p-value': [p_BMP, p_BMP_adj]
    }

    return pd.DataFrame(results).set_index('Test')


In [ ]:
#Fama-French estimation

#OLS Regression using CAPM


# Merge stock returns with Fama-French factors
combined_returns = stock_returns.join(factors, how='inner')
combined_returns.index = pd.to_datetime(combined_returns.index)

event_date = pd.to_datetime(event_date)
window = 5
event_dates = pd.date_range(event_date - timedelta(days=window), event_date + timedelta(days=window))
print(event_dates)
# Define estimation and event windows
estimation_data = combined_returns[
    (~combined_returns.index.isin(event_dates)) &
    (combined_returns.index < event_date - timedelta(days=window))
]
event_data = combined_returns[combined_returns.index.isin(event_dates)]

# Temporary dictionaries to collect results
abnormal_event_dict = {}
abnormal_estimation_dict = {}

# Loop through stocks
for stock in stock_returns.columns:
    y = estimation_data[stock] - estimation_data['RF']
    X = sm.add_constant(estimation_data[['Mkt-RF', 'SMB', 'HML']])
    model = sm.OLS(y, X).fit()

    # Predict for event window
    X_event = sm.add_constant(event_data[['Mkt-RF', 'SMB', 'HML']])
    expected_event = model.predict(X_event)
    realized_event = event_data[stock] - event_data['RF']
    abnormal_event_dict[stock] = realized_event - expected_event

    # Predict for estimation window
    expected_estimation = model.predict(X)
    realized_estimation = estimation_data[stock] - estimation_data['RF']
    abnormal_estimation_dict[stock] = realized_estimation - expected_estimation

# Build DataFrames at once to avoid fragmentation
abnormal_returns_event = pd.DataFrame(abnormal_event_dict, index=event_data.index)
abnormal_returns_estimation = pd.DataFrame(abnormal_estimation_dict, index=estimation_data.index)

ticker_to_sector = stock_attributes['Industry'].to_dict()

# Create MultiIndex for columns: (Sector, Ticker)
new_columns = [
    (ticker_to_sector.get(ticker, 'Unknown'), ticker)
    for ticker in abnormal_returns_estimation.columns
]

# Set the new MultiIndex
abnormal_returns_estimation.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])
abnormal_returns_event.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])


In [ ]:

# Example: map of tickers to sectors
# attributes_df: columns = ['Ticker', 'Sector']
ticker_to_sector = stock_attributes['Industry'].to_dict()

# Create MultiIndex for columns: (Sector, Ticker)
new_columns = [
    (ticker_to_sector.get(ticker, 'Unknown'), ticker)
    for ticker in abnormal_returns_estimation.columns
]

# Set the new MultiIndex
abnormal_returns_estimation.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])


In [ ]:


for sector in abnormal_returns_estimation.columns.levels[0]:
    print(f"\n{sector}")
    print(f"CAAR: {abnormal_returns_event[sector].sum().mean()}")
    print(run_patell_test(abnormal_returns_estimation[sector],abnormal_returns_event[sector],estimation_data,event_data,event_date,'Market'))
    print(run_bmp_test(abnormal_returns_estimation[sector],abnormal_returns_event[sector],estimation_data,event_data,'Market'))


